In [4]:
import duckdb
import pandas as pd
import sqlite3

In [5]:
# Here connect to the database file
conn = duckdb.connect("analytics.duckdb", read_only = True)

# Here we check the table names
table_name_query_df = conn.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
""").df()

print("Here are the tables names from the database:")
print('-'*40)
print(table_name_query_df)

Here are the tables names from the database:
----------------------------------------
                 table_name
0           auto_sales_2022
1  kob_industry_total_sales
2                  outliers
3           sales_above_10b
4   sale_distribution_naics
5           seasonality_mom
6     top_industry_mom_2019
7     top_industry_mom_2020
8     top_industry_mom_2021
9     top_industry_mom_2022


## Business Questions

#### **1. The Top performing industries in terms of sales for the year 2021 and how do their sales compare month-over-month**

In [6]:
# Here we get the top performing industry for the year 2021
query1 = """
WITH TopIndustry AS (
     SELECT industry
     FROM "top_industry_mom_2022"
     GROUP BY industry
     ORDER BY SUM(total_sales) DESC
     LIMIT 1
)

SELECT industry, month, total_sales,
       LAG(total_sales) OVER (PARTITION BY industry ORDER BY month) AS sales_prev_month,
       ROUND(
             (total_sales - LAG(total_sales) OVER (PARTITION BY industry ORDER BY month)) * 100, 2
       ) AS mom_growth_percentage

FROM "top_industry_mom_2022"
WHERE industry IN (SELECT industry FROM TopIndustry)
ORDER BY month;
"""

industry_best_performers_sales_2022 = conn.execute(query1).df()
industry_best_performers_sales_2022.head()

,industry,month,total_sales,sales_prev_month,mom_growth_percentage
0,Automotive,1,420473,<NA>,<NA>
1,Automotive,2,431998,420473,1152500
2,Automotive,3,514582,431998,8258400
3,Automotive,4,504116,514582,-1046600
4,Automotive,5,483482,504116,-2063400


In [7]:
# what does the NA sales per month indicate, since there's no previous record for month 0 could be the record december of the previous year
# Using COALESCE () method replaces the NA value with the default (0) or fallback value

# Here we get the top performing industry for the year 2021 but since we got NA value for the first record,
# improve the query using COALESCE () method
query1 = """
WITH TopIndustry AS (
     SELECT industry
     FROM "top_industry_mom_2022"
     GROUP BY industry
     ORDER BY SUM(total_sales) DESC
     LIMIT 1
)

SELECT industry, month, total_sales,
       COALESCE(LAG(total_sales) OVER (PARTITION BY industry ORDER BY month), 0) AS sales_prev_month,
       COALESCE(
       ROUND(
             (total_sales - LAG(total_sales) OVER (PARTITION BY industry ORDER BY month)) * 100, 2), 0
       ) AS mom_growth_percentage

FROM "top_industry_mom_2022"
WHERE industry IN (SELECT industry FROM TopIndustry)
ORDER BY month;
"""

industry_best_performers_sales_2022_rf = conn.execute(query1).df()
industry_best_performers_sales_2022_rf.head()

,industry,month,total_sales,sales_prev_month,mom_growth_percentage
0,Automotive,1,420473,0,0
1,Automotive,2,431998,420473,1152500
2,Automotive,3,514582,431998,8258400
3,Automotive,4,504116,514582,-1046600
4,Automotive,5,483482,504116,-2063400


#### **2. The Top performing industries in terms of sales for the year 2021 and how do their sales compare month-over-month**

In [8]:
query2 = """
WITH TopIndustry AS (
     SELECT industry
     FROM "top_industry_mom_2021"
     GROUP BY industry
     ORDER BY SUM(total_sales) DESC
     LIMIT 1
)

SELECT industry, month, total_sales,
       COALESCE(LAG(total_sales) OVER (PARTITION BY industry ORDER BY month), 0) AS sales_prev_month,
       COALESCE(
       ROUND(
             (total_sales - LAG(total_sales) OVER (PARTITION BY industry ORDER BY month)) * 100, 2), 0
       ) AS mom_growth_percentage

FROM "top_industry_mom_2021"
WHERE industry IN (SELECT industry FROM TopIndustry)
ORDER BY month;
"""

industry_best_performers_sales_2021 = conn.execute(query2).df()
industry_best_performers_sales_2021.head()

,industry,month,total_sales,sales_prev_month,mom_growth_percentage
0,Automotive,1,375492,0,0
1,Automotive,2,369166,375492,-632600
2,Automotive,3,524244,369166,15507800
3,Automotive,4,512810,524244,-1143400
4,Automotive,5,508814,512810,-399600


#### **3. The Top performing industries in terms of sales for the year 2020 and how do their sales compare month-over-month**

In [9]:
query3 = """
WITH TopIndustry AS (
     SELECT industry
     FROM "top_industry_mom_2020"
     GROUP BY industry
     ORDER BY SUM(total_sales) DESC
     LIMIT 1
)

SELECT industry, month, total_sales,
       COALESCE(LAG(total_sales) OVER (PARTITION BY industry ORDER BY month), 0) AS sales_prev_month,
       COALESCE(
       ROUND(
             (total_sales - LAG(total_sales) OVER (PARTITION BY industry ORDER BY month)) * 100, 2), 0
       ) AS mom_growth_percentage

FROM "top_industry_mom_2020"
WHERE industry IN (SELECT industry FROM TopIndustry)
ORDER BY month;
"""

industry_best_performers_sales_2020 = conn.execute(query3).df()
industry_best_performers_sales_2020.head()

,industry,month,total_sales,sales_prev_month,mom_growth_percentage
0,Automotive,1,342994,0,0
1,Automotive,2,358432,342994,1543800
2,Automotive,3,295174,358432,-6325800
3,Automotive,4,245248,295174,-4992600
4,Automotive,5,380186,245248,13493800


#### **4. The Top performing industries in terms of sales for the year 2019 and how do their sales compare month-over-month**

In [10]:
query4 = """
WITH TopIndustry AS (
     SELECT industry
     FROM "top_industry_mom_2019"
     GROUP BY industry
     ORDER BY SUM(total_sales) DESC
     LIMIT 1
)

SELECT industry, month, total_sales,
       COALESCE(LAG(total_sales) OVER (PARTITION BY industry ORDER BY month), 0) AS sales_prev_month,
       COALESCE(
       ROUND(
             (total_sales - LAG(total_sales) OVER (PARTITION BY industry ORDER BY month)) * 100, 2), 0
       ) AS mom_growth_percentage

FROM "top_industry_mom_2019"
WHERE industry IN (SELECT industry FROM TopIndustry)
ORDER BY month;
"""

industry_best_performers_sales_2019 = conn.execute(query4).df()
industry_best_performers_sales_2019.head()

,industry,month,total_sales,sales_prev_month,mom_growth_percentage
0,Automotive,1,329164,0,0
1,Automotive,2,335628,329164,646400
2,Automotive,3,411614,335628,7598600
3,Automotive,4,383028,411614,-2858600
4,Automotive,5,414218,383028,3119000


#### **Business Question 2:**
#### Which specific kind of businesses contribute the most to the total sales and how does their performance vary across industries?

In [11]:
# query5 = """
# SELECT kind_of_business, industry,
#        SUM(total_sales) AS total_sales
# FROM kob_industry_total_sales
# GROUP BY kind_of_business, industry
# ORDER BY total_sales
# DESC
# LIMIT 15;
# """
# business_most_total_sales = conn.execute(query5).df()
# business_most_total_sales

In [12]:
# Here we first calculate the total overall sales across all businesses then rank each industry based on their total market sales
query2_1 = """
WITH total_market_sales AS (
     SELECT SUM(total_sales) AS grand_sales
     FROM "kob_industry_total_sales"
),

kob_performance AS (
     SELECT industry, kind_of_business,
            SUM(total_sales) AS kob_total_sales,
            DENSE_RANK() OVER (PARTITION BY industry ORDER BY SUM(total_sales) DESC) AS rank_within_industry
     FROM "kob_industry_total_sales"
     GROUP BY industry, kind_of_business
)
SELECT
     kp.industry,
     kp.kind_of_business,
     kp.kob_total_sales,
     ROUND((kp.kob_total_sales / tms.grand_sales) * 100, 2) AS percentage_total_market_sales,
     kp.rank_within_industry

FROM kob_performance AS kp
CROSS JOIN total_market_sales AS tms
WHERE kp.rank_within_industry = 1
ORDER BY  kp.kob_total_sales DESC
LIMIT 10;

"""
business_most_sales = conn.execute(query2_1).df()
business_most_sales.head()

,industry,kind_of_business,kob_total_sales,percentage_total_market_sales,rank_within_industry
0,Automotive,Motor vehicle and parts dealers,14531121.0,7.57,1
1,Food & Beverage,Food and beverage stores,9451004.0,4.92,1
2,General Merchandise,General merchandise stores,9038491.0,4.71,1
3,Restaurants & Bars,Food services and drinking places,8635492.0,4.50,1
4,Miscellaneous,Nonstore retailers,8511856.0,4.43,1


In [13]:
# Keeping the rank_within_industry = 1 hides the rank 2,3 which might lead to lose of insights into whether an industry is monopolized by single giant business
# type or maybe spreads evenly across several business types

# Lacks the comparative metrics; performance variation which requires comparing metrics like average sales per business growth rate or even the market concentration rather than just sorting out total sales across different industry sectors.


In [14]:
# Here is the query for the improved version with performance variations across industry.
query2_1 = """
-- Here we get the total sales per industry--
WITH total_market_sales AS (
     SELECT industry,
     SUM(total_sales) AS grand_sales
     FROM "kob_industry_total_sales"
     GROUP BY industry
),

-- Find the grand sales across all sectors --
overall_market_performance AS (
     SELECT SUM(total_sales) AS grand_total_sales
     FROM "kob_industry_total_sales"
),

-- Then rank the businesses within each industry --
ranked_kob AS (
     SELECT
           k.industry, k.kind_of_business,
           SUM(k.total_sales) AS kob_grand_sales,
           DENSE_RANK() OVER (PARTITION BY k.industry ORDER BY SUM(k.total_sales) DESC) AS rank_within_industry
     FROM "kob_industry_total_sales" k
     GROUP BY k.industry, k.kind_of_business
)

SELECT
     r.industry, r.kind_of_business AS top_ranked_business,
     r.kob_grand_sales,

     -- Getting the market contribution --
     ROUND((r.kob_grand_sales / omp.grand_total_sales) * 100, 2) AS percentage_total_market_sales,

     -- Check the variance for the industry-dominance--
     ROUND((r.kob_grand_sales / tms.grand_sales) * 100, 2) AS percentage_share_within_industry

FROM ranked_kob r
JOIN total_market_sales tms ON r.industry = tms.industry
CROSS JOIN overall_market_performance omp
WHERE r.rank_within_industry = 1
ORDER BY r.kob_grand_sales
DESC
LIMIT 10;

"""
top_performing_business_sales = conn.execute(query2_1).df()
top_performing_business_sales

,industry,top_ranked_business,kob_grand_sales,percentage_total_market_sales,percentage_share_within_industry
0,Automotive,Motor vehicle and parts dealers,14531121.0,7.57,26.99
1,Food & Beverage,Food and beverage stores,9451004.0,4.92,28.83
2,General Merchandise,General merchandise stores,9038491.0,4.71,42.96
3,Restaurants & Bars,Food services and drinking places,8635492.0,4.50,37.81
4,Miscellaneous,Nonstore retailers,8511856.0,4.43,82.03
5,Home Goods & Electronics,Electronic shopping and mail-order houses,7351761.0,3.83,46.99
6,Fuel & Gasoline,Gasoline stations,6811521.0,3.55,93.57
7,Home Goods & Building Supplies,Building mat. and garden equip. and supplies d...,4586452.0,2.39,50.55
8,Health & Personal Care,Health and personal care stores,4149035.0,2.16,54.47
9,Fashion & Accessories,Clothing and clothing access. stores,3302936.0,1.72,38.50


#### **Business Question 3:**
#### Is there any seasonality in sales for specific industries, and how do they perform month-over-month?

In [15]:
query3_s = """
WITH monthly_sales AS (
     SELECT industry,
            CAST(month AS INT) AS month_num,
            SUM(total_sales) AS monthly_sales
     FROM "seasonality_mom"
     GROUP BY industry, month_num
)

SELECT industry, month_num, monthly_sales,
       COALESCE(LAG(monthly_sales) OVER (PARTITION BY industry ORDER BY month_num), 0) AS prev_month_sales,
       COALESCE(
       ROUND(
            ((monthly_sales - LAG(monthly_sales) OVER (PARTITION BY industry ORDER BY month_num))
            / LAG(monthly_sales) OVER (PARTITION BY industry ORDER BY month_num)) * 100, 2), 0
       ) AS percentage_growth_mom

FROM monthly_sales
ORDER BY industry, month_num;

"""
seasonality_trends_industry = conn.execute(query3_s).df()
seasonality_trends_industry.head()


,industry,month_num,monthly_sales,prev_month_sales,percentage_growth_mom
0,Automotive,1,3867979.0,0.0,0.00
1,Automotive,2,4037466.0,3867979.0,4.38
2,Automotive,3,4759220.0,4037466.0,17.88
3,Automotive,4,4471542.0,4759220.0,-6.04
4,Automotive,5,4763342.0,4471542.0,6.53


#### **Business Question 4:**
#### How does the sales distribution vary among industries based on their North American Industry Classification System (NAICS) codes?

In [16]:
query4_distribution = """
WITH market_sales AS (
     SELECT SUM(total_sales) AS grand_total_sales
     FROM "sale_distribution_naics"
)

SELECT
     sdn.naics_code, sdn.industry,
     SUM(sdn.total_sales) AS total_sales,
     ROUND((SUM(sdn.total_sales) / ms.grand_total_sales) * 100, 2) AS percentage_market_shares,
     DENSE_RANK() OVER (ORDER BY SUM(sdn.total_sales) DESC) AS rank_in_sales

 FROM "sale_distribution_naics" sdn
 CROSS JOIN market_sales ms
 GROUP BY sdn.naics_code, sdn.industry, ms.grand_total_sales
 ;
"""
naics_distribution_sales = conn.execute(query4_distribution).df()
naics_distribution_sales.head(10)

,naics_code,industry,total_sales,percentage_market_shares,rank_in_sales
0,441,Automotive,14531121.0,7.57,1
1,"44,114,412",Automotive,13310756.0,6.93,2
2,4411,Automotive,12389347.0,6.45,3
3,44111,Automotive,10980583.0,5.72,4
4,445,Food & Beverage,9451004.0,4.92,5
5,452,General Merchandise,9038491.0,4.71,6
6,722,Restaurants & Bars,8635492.0,4.50,7
7,454,Miscellaneous,8511856.0,4.43,8
8,4451,Food & Beverage,8463197.0,4.41,9
9,44511,Food & Beverage,8092220.0,4.21,10


#### **Business Question 5:**
#### Are there any outliers or significant changes in sales for specific industries during particular months or years?

In [17]:
query5_outliers_sales = """
WITH industry_stats AS (
     SELECT industry, year, month,
            TRY_CAST (sales AS DOUBLE) AS total_sales,
            ROUND(AVG(total_sales) OVER (PARTITION BY industry), 2) AS average_sales,
            ROUND((total_sales - AVG(total_sales) OVER W) / NULLIF(STDDEV(total_sales) OVER w, 0), 2) AS z_score
     FROM outliers
     WINDOW w AS (PARTITION BY industry)
)

SELECT *,
       CASE WHEN z_score > 0 THEN 'Spike (+2σ)' ELSE 'Drop (-2σ)' END AS anomaly_type

FROM industry_stats
WHERE ABS(z_score) > 2
ORDER BY industry, year, month;

"""
industry_outliers_sales = conn.execute(query5_outliers_sales).df()
industry_outliers_sales.head()


,industry,year,month,total_sales,average_sales,z_score,anomaly_type
0,Automotive,2021,3,141659.0,57522.37,2.19,Spike (+2σ)
1,Automotive,2021,4,138891.0,57522.37,2.12,Spike (+2σ)
2,Automotive,2021,5,137600.0,57522.37,2.09,Spike (+2σ)
3,Automotive,2022,3,140108.0,57522.37,2.15,Spike (+2σ)
4,Automotive,2022,4,137296.0,57522.37,2.08,Spike (+2σ)


#### **Business Question:**
#### Which businesses all-time average sale was above 10 billion dollars?

In [18]:
query_sales_10b = """
SELECT kind_of_business,
       average_sale AS average_sales

FROM sales_above_10b
GROUP BY kind_of_business, average_sales
-- 10 billion dollars in cents (1 dollar = 100 cents)
HAVING average_sales > 10000

"""
sales_above_10b_dollars = conn.execute(query_sales_10b).df()
sales_above_10b_dollars.head()

,kind_of_business,average_sales
0,Electronic shopping and mail-order houses,47126.673077
1,Department stores,12864.000000
2,Pharmacies and drug stores,22227.852564
3,Supermarkets and other grocery (except conveni...,51873.205128
4,Clothing and clothing access. stores,21172.666667


#### Which kind of businesses within the automotive industry had the highest sales revenue for 2022?

In [19]:
query_top_automotive_ind_2022 = """
WITH top_industry_sales_2022 AS (
     SELECT
            kob.kind_of_business, top_2022.year,
            kob.industry,
            SUM(kob.total_sales) AS total_sales
     FROM kob_industry_total_sales kob
     JOIN top_industry_mom_2022 top_2022
          ON kob.industry = top_2022.industry
     GROUP BY kob.kind_of_business, kob.industry, top_2022.year
)
SELECT *
FROM top_industry_sales_2022
WHERE industry = 'Automotive' AND year = 2022
ORDER BY total_sales DESC;

"""
sales_automotive_industry_2022 = conn.execute(query_top_automotive_ind_2022).df()
sales_automotive_industry_2022.head()

,kind_of_business,year,industry,total_sales
0,Motor vehicle and parts dealers,2022,Automotive,174373452.0
1,Automobile and other motor vehicle dealers,2022,Automotive,159729072.0
2,Automobile dealers,2022,Automotive,148672164.0
3,New car dealers,2022,Automotive,131766996.0
4,Used car dealers,2022,Automotive,16905168.0


#### Which kind of business within the automotive industry had the highest sales revenue for 2020?

In [20]:
query_top_automotive_ind_2020 = """
WITH top_industry_sales_2020 AS (
     SELECT
           kob.kind_of_business, top_2020.year,
           kob.industry,
           SUM(kob.total_sales) AS total_sales
     FROM kob_industry_total_sales kob
     JOIN top_industry_mom_2020 top_2020
          ON kob.industry = top_2020.industry
     GROUP BY kob.kind_of_business, kob.industry, top_2020.year

)
SELECT *
FROM top_industry_sales_2020
WHERE industry = 'Automotive' AND year = 2020
ORDER BY total_sales DESC;

"""
sales_automotive_industry_2020 = conn.execute(query_top_automotive_ind_2020).df()
sales_automotive_industry_2020.head()

,kind_of_business,year,industry,total_sales
0,Motor vehicle and parts dealers,2020,Automotive,174373452.0
1,Automobile and other motor vehicle dealers,2020,Automotive,159729072.0
2,Automobile dealers,2020,Automotive,148672164.0
3,New car dealers,2020,Automotive,131766996.0
4,Used car dealers,2020,Automotive,16905168.0


In [21]:
table_name_query_df

,table_name
0,auto_sales_2022
1,kob_industry_total_sales
2,outliers
3,sales_above_10b
4,sale_distribution_naics
5,seasonality_mom
6,top_industry_mom_2019
7,top_industry_mom_2020
8,top_industry_mom_2021
9,top_industry_mom_2022


#### What is the contribution percentage of each business in the automotive industry this year?

In [22]:
query_contribution_business_sales_2022 = """
WITH perc_automotive_indu_sales AS (
     SELECT
           kob.kind_of_business, top_2022.year,
           kob.industry,
           SUM(kob.total_sales) AS grand_sales_business
    FROM kob_industry_total_sales kob
    JOIN top_industry_mom_2022 top_2022
         ON kob.industry = top_2022.industry
    WHERE kob_industry = 'Automotive' AND top_2022.year = 2022
    GROUP BY kob.kind_of_business

),
industry_grand_sales AS (
     SELECT SUM(total_sales) AS total_automotive_sales
     FROM auto_sales_2022
)

SELECT
      as_2022.kind_of_business,
      as_2022.total_sales,
      ROUND((as_2022.total_sales / igs.total_automotive_sales) * 100, 2) AS percentage_contribution
FROM auto_sales_2022 AS as_2022
CROSS JOIN industry_grand_sales AS igs
ORDER BY as_2022.total_sales DESC;

"""
industry_percentage_sales_performance_2022 = conn.execute(query_contribution_business_sales_2022).df()
industry_percentage_sales_performance_2022.head()

,kind_of_business,total_sales,percentage_contribution
0,Motor vehicle and parts dealers,1527929,27.17
1,Automobile and other motor vehicle dealers,1403735,24.96
2,Automobile dealers,1284189,22.83
3,New car dealers,1123378,19.97
4,Used car dealers,160811,2.86


#### What are the year-over-year growth rates for each industry per year?

In [23]:
# UNION ALL: Stacks yearly data (2019–2022) into a unified time series, casting the explicit year literal as an integer column.
#
# PARTITION BY industry ORDER BY year: Isolates each industry independently so that LAG() compares 2020 vs. 2019, 2021 vs. 2020, and 2022 vs. 2021.
#
# NULLIF(prior_year_sales, 0): Prevents division-by-zero runtime exceptions for new industries or zero-baseline periods.
#
# COALESCE(..., 0): Handles the baseline year (2019), replacing the initial NULL with 0 for clean reporting.

In [24]:
query_yearly_growth = """
WITH annual_industry_sales AS (
     SELECT industry, 2019 AS year, SUM(total_sales) AS yearly_sales FROM top_industry_mom_2019 GROUP BY industry
     UNION ALL
     SELECT industry, 2020 AS year, SUM(total_sales) AS yearly_sales FROM top_industry_mom_2020 GROUP BY industry
     UNION ALL
     SELECT industry, 2021 AS year, SUM(total_sales) AS yearly_sales FROM top_industry_mom_2021 GROUP BY industry
     UNION ALL
     SELECT industry, 2022 AS year, SUM(total_sales) AS yearly_sales FROM top_industry_mom_2022 GROUP BY industry

),

yearly_growth_arithmetic AS (
     SELECT industry, year, yearly_sales,
            LAG(yearly_sales) OVER (PARTITION BY industry ORDER BY year ASC) AS previous_years_sales
     FROM annual_industry_sales
)

SELECT industry, year, yearly_sales,
       previous_years_sales,
       COALESCE(
          ROUND(
          (yearly_sales - previous_years_sales) / NULLIF(previous_years_sales, 0) * 100, 2
          ), 0
       ) AS yearly_growth

FROM yearly_growth_arithmetic
ORDER BY industry, year
;
"""

yearly_growth_rate_industry = conn.execute(query_yearly_growth).df()
yearly_growth_rate_industry.head(10)

,industry,year,yearly_sales,previous_years_sales,yearly_growth
0,Automotive,2019,4606586.0,NaN,0.00
1,Automotive,2020,4462952.0,4606586.0,-3.12
2,Automotive,2021,5475988.0,4462952.0,22.70
3,Automotive,2022,5624235.0,5475988.0,2.71


In [25]:
# When calculating year-over-year growth using window functions like LAG(), The first year recorded for each industry group results into NaN for the previous year sales and yearly growth, such case is because there's no baseline data prior to the records from the data